In [1]:
import os
import torch
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split,
    WeightedRandomSampler
)

from torchvision import transforms
from torchvision import models

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", device)

Using Device: cpu


In [3]:
image_dir = r"D:\medical paper\ISIC_2024_Permissive_Training_Input"

csv_path = r"D:\medical paper\ISIC_2024_Permissive_Training_GroundTruth.csv"

In [4]:
df = pd.read_csv(csv_path)

print(df.head())

print(df.shape)

        isic_id  malignant
0  ISIC_0015670        0.0
1  ISIC_0015845        0.0
2  ISIC_0015864        0.0
3  ISIC_0015902        0.0
4  ISIC_0024200        0.0
(217477, 2)


In [5]:
image_column = "isic_id"

label_column = "malignant"

df = df.dropna(subset=[label_column])

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(
    df[label_column]
)

class_names = label_encoder.classes_

print("Classes:", class_names)

Classes: [0. 1.]


In [6]:
print(df["label"].value_counts())

print(
    df["label"].value_counts(normalize=True) * 100
)

label
0    217183
1       294
Name: count, dtype: int64
label
0    99.864813
1     0.135187
Name: proportion, dtype: float64


In [23]:
# ==========================================
# CREATE BALANCED DATASET
# ==========================================

malignant_df = df[
    df["label"] == 1
]

benign_df = df[
    df["label"] == 0
].sample(
    n=5000,
    random_state=42
)

balanced_df = pd.concat([
    benign_df,
    malignant_df
])

balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nBalanced Dataset Distribution:")

print(
    balanced_df["label"].value_counts()
)


Balanced Dataset Distribution:
label
0    5000
1     294
Name: count, dtype: int64


In [25]:
class ISICDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform=None
    ):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.image_dir = image_dir

        self.transform = transform

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_id = str(row["isic_id"])

        image_path = os.path.join(
            self.image_dir,
            image_id + ".jpg"
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        label = int(row["label"])

        if self.transform:

            image = self.transform(image)

        return image, label

In [27]:
full_dataset = ISICDataset(
    dataframe=balanced_df,
    image_dir=image_dir,
    transform=train_transform
)

print("Total Images:", len(full_dataset))

Total Images: 5294


In [28]:
dataset_size = len(full_dataset)

train_size = int(0.70 * dataset_size)

val_size = int(0.15 * dataset_size)

test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 3705
Validation: 794
Test: 795


In [29]:
train_labels = []

for idx in train_dataset.indices:

    train_labels.append(
        full_dataset.dataframe.iloc[idx]["label"]
    )

from collections import Counter

class_count = Counter(train_labels)

print(class_count)

Counter({0: 3505, 1: 200})


In [30]:
weights = []

for label in train_labels:

    weights.append(
        1.0 / class_count[label]
    )

sampler = WeightedRandomSampler(
    weights,
    num_samples=len(weights),
    replacement=True
)

In [31]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("DataLoaders Ready")

DataLoaders Ready


In [32]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    2
)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=2, bias=True)


In [33]:
class_weights = torch.tensor(
    [
        len(train_labels)/(2*class_count[0]),
        len(train_labels)/(2*class_count[1])
    ],
    dtype=torch.float
).to(device)

print(class_weights)

tensor([0.5285, 9.2625])


In [34]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [17]:
EPOCHS = 15

best_val_acc = 0

train_losses = []
val_losses = []

train_accs = []
val_accs = []

In [35]:
print("Total Images:", len(full_dataset))
print(class_count)
print(class_weights)

Total Images: 5294
Counter({0: 3505, 1: 200})
tensor([0.5285, 9.2625])


In [36]:
from tqdm import tqdm

EPOCHS = 15

best_val_acc = 0

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(
            outputs,
            1
        )

        train_total += labels.size(0)

        train_correct += (
            predicted == labels
        ).sum().item()

        train_acc = (
            100 * train_correct /
            train_total
        )

        progress_bar.set_postfix({
            "Loss": f"{loss.item():.4f}",
            "Acc": f"{train_acc:.2f}%"
        })

    # Validation

    model.eval()

    val_correct = 0
    val_total = 0
    val_loss = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

            _, predicted = torch.max(
                outputs,
                1
            )

            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()

    train_accuracy = (
        100 * train_correct /
        train_total
    )

    val_accuracy = (
        100 * val_correct /
        val_total
    )

    print("\n" + "="*60)

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Accuracy: "
        f"{train_accuracy:.2f}%"
    )

    print(
        f"Validation Accuracy: "
        f"{val_accuracy:.2f}%"
    )

    print("="*60)

    # Save Best Model

    if val_accuracy > best_val_acc:

        best_val_acc = val_accuracy

        torch.save(
            model.state_dict(),
            "best_resnet18_skin_lesion.pth"
        )

        print(
            "✅ Best Model Saved"
        )

Epoch 1/15: 100%|██████████| 116/116 [05:35<00:00,  2.89s/it, Loss=0.1081, Acc=66.18%]



Epoch 1/15
Train Accuracy: 66.18%
Validation Accuracy: 76.07%
✅ Best Model Saved


Epoch 2/15: 100%|██████████| 116/116 [05:19<00:00,  2.76s/it, Loss=0.0617, Acc=84.83%]



Epoch 2/15
Train Accuracy: 84.83%
Validation Accuracy: 72.29%


Epoch 3/15: 100%|██████████| 116/116 [04:59<00:00,  2.59s/it, Loss=0.0270, Acc=89.61%]


...


Epoch 13/15
Train Accuracy: 97.46%
Validation Accuracy: 90.43%


Epoch 14/15: 100%|██████████| 116/116 [04:29<00:00,  2.32s/it, Loss=0.0078, Acc=97.62%]



Epoch 14/15
Train Accuracy: 97.62%
Validation Accuracy: 91.94%


Epoch 15/15: 100%|██████████| 116/116 [04:30<00:00,  2.33s/it, Loss=0.0015, Acc=98.97%]



Epoch 15/15
Train Accuracy: 98.97%
Validation Accuracy: 92.82%


In [37]:
model.load_state_dict(
    torch.load(
        "best_resnet18_skin_lesion.pth"
    )
)

model.eval()

print("Best Model Loaded")

Best Model Loaded


In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(
            outputs,
            1
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

print(
    "Accuracy:",
    accuracy_score(
        all_labels,
        all_preds
    )
)

print(
    "Precision:",
    precision_score(
        all_labels,
        all_preds
    )
)

print(
    "Recall:",
    recall_score(
        all_labels,
        all_preds
    )
)

print(
    "F1 Score:",
    f1_score(
        all_labels,
        all_preds
    )
)

print(
    "\nConfusion Matrix:\n",
    confusion_matrix(
        all_labels,
        all_preds
    )
)

Accuracy: 0.9610062893081761
Precision: 0.52
Recall: 0.7878787878787878
F1 Score: 0.6265060240963856

Confusion Matrix:
 [[738  24]
 [  7  26]]


In [39]:
malignant_samples = df[df["label"] == 1]

print(malignant_samples.head(20))

            isic_id  malignant  label
544    ISIC_0096034        1.0      1
714    ISIC_0104229        1.0      1
1023   ISIC_0119495        1.0      1
1928   ISIC_0157834        1.0      1
2662   ISIC_0190307        1.0      1
3132   ISIC_0211092        1.0      1
4457   ISIC_0275647        1.0      1
4517   ISIC_0279372        1.0      1
4692   ISIC_0287900        1.0      1
4824   ISIC_0293670        1.0      1
5618   ISIC_0330452        1.0      1
6888   ISIC_0386460        1.0      1
7032   ISIC_0392749        1.0      1
7884   ISIC_0429895        1.0      1
8393   ISIC_0452332        1.0      1
8730   ISIC_0467830        1.0      1
9876   ISIC_0521019        1.0      1
10021  ISIC_0528190        1.0      1
10579  ISIC_0554252        1.0      1
10826  ISIC_0565021        1.0      1


In [40]:
malignant_samples = df[df["label"] == 1]

print(
    malignant_samples[
        ["isic_id", "malignant"]
    ].head(10)
)

           isic_id  malignant
544   ISIC_0096034        1.0
714   ISIC_0104229        1.0
1023  ISIC_0119495        1.0
1928  ISIC_0157834        1.0
2662  ISIC_0190307        1.0
3132  ISIC_0211092        1.0
4457  ISIC_0275647        1.0
4517  ISIC_0279372        1.0
4692  ISIC_0287900        1.0
4824  ISIC_0293670        1.0


In [41]:
malignant_samples = df[df["label"] == 1]

malignant_ids = malignant_samples["isic_id"].tolist()[:10]

print("\nTop 10 Malignant Images:\n")

for idx, image_id in enumerate(malignant_ids, start=1):
    print(f"{idx}. {image_id}")



Top 10 Malignant Images:

1. ISIC_0096034
2. ISIC_0104229
3. ISIC_0119495
4. ISIC_0157834
5. ISIC_0190307
6. ISIC_0211092
7. ISIC_0275647
8. ISIC_0279372
9. ISIC_0287900
10. ISIC_0293670
